In [34]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np
import text_normalizer as tn
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [35]:
data = fetch_20newsgroups(subset='all', shuffle=True,
                          remove=('headers', 'footers', 'quotes'))
data_labels_map = dict(enumerate(data.target_names))

In [36]:
corpus, target_labels, target_names = (data.data, data.target, 
                                       [data_labels_map[label] for label in data.target])
data_df = pd.DataFrame({'Article': corpus, 'Target Label': target_labels, 'Target Name': target_names})
print(data_df.shape)
data_df.head(10)

(18846, 3)


,Article,Target Label,Target Name
0,\n\nI am sure some bashers of Pens fans are pr...,10,rec.sport.hockey
1,My brother is in the market for a high-perform...,3,comp.sys.ibm.pc.hardware
2,\n\n\n\n\tFinally you said what you dream abou...,17,talk.politics.mideast
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,3,comp.sys.ibm.pc.hardware
4,1) I have an old Jasmine drive which I cann...,4,comp.sys.mac.hardware
5,\n\nBack in high school I worked as a lab assi...,12,sci.electronics
6,\n\nAE is in Dallas...try 214/241-6060 or 214/...,4,comp.sys.mac.hardware
7,"\n[stuff deleted]\n\nOk, here's the solution t...",10,rec.sport.hockey
8,"\n\n\nYeah, it's the second one. And I believ...",10,rec.sport.hockey
9,\nIf a Christian means someone who believes in...,19,talk.religion.misc


In [37]:
total_nulls = data_df[data_df.Article.str.strip() == ''].shape[0]
print("Empty documents:", total_nulls)

Empty documents: 515


In [38]:
data_df = data_df[~(data_df.Article.str.strip() == '')]
data_df.shape

(18331, 3)

In [39]:
import nltk
stopword_list = nltk.corpus.stopwords.words('english')
# just to keep negation if any in bi-grams
stopword_list.remove('no')
stopword_list.remove('not')

# normalize our corpus
norm_corpus = tn.normalize_corpus(corpus=data_df['Article'], html_stripping=True, contraction_expansion=True, 
                                  accented_char_removal=True, text_lower_case=True, text_lemmatization=True, 
                                  text_stemming=False, special_char_removal=True, remove_digits=True,
                                  stopword_removal=True, stopwords=stopword_list)
data_df['Clean Article'] = norm_corpus

In [40]:
data_df = data_df[['Article', 'Clean Article', 'Target Label', 'Target Name']]
data_df.head(10)

,Article,Clean Article,Target Label,Target Name
0,\n\nI am sure some bashers of Pens fans are pr...,sure basher pens fan pretty confused lack kind...,10,rec.sport.hockey
1,My brother is in the market for a high-perform...,brother market high performance video card sup...,3,comp.sys.ibm.pc.hardware
2,\n\n\n\n\tFinally you said what you dream abou...,finally say dream mediterranean new area great...,17,talk.politics.mideast
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,think scsi card dma transfer not disk scsi car...,3,comp.sys.ibm.pc.hardware
4,1) I have an old Jasmine drive which I cann...,old jasmine drive not use new system understan...,4,comp.sys.mac.hardware
5,\n\nBack in high school I worked as a lab assi...,back high school work lab assistant bunch expe...,12,sci.electronics
6,\n\nAE is in Dallas...try 214/241-6060 or 214/...,ae dallas try tech support may line one get start,4,comp.sys.mac.hardware
7,"\n[stuff deleted]\n\nOk, here's the solution t...",stuff delete ok solution problem move canada y...,10,rec.sport.hockey
8,"\n\n\nYeah, it's the second one. And I believ...",yeah second one believe price try get good loo...,10,rec.sport.hockey
9,\nIf a Christian means someone who believes in...,christian mean someone believe divinity jesus ...,19,talk.religion.misc


In [41]:
data_df = data_df.replace(r'^(\s?)+$', np.nan, regex=True)
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18331 entries, 0 to 18845
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Article        18331 non-null  object
 1   Clean Article  18300 non-null  object
 2   Target Label   18331 non-null  int64 
 3   Target Name    18331 non-null  object
dtypes: int64(1), object(3)
memory usage: 716.1+ KB


In [42]:
data_df = data_df.dropna().reset_index(drop=True)
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18300 entries, 0 to 18299
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Article        18300 non-null  object
 1   Clean Article  18300 non-null  object
 2   Target Label   18300 non-null  int64 
 3   Target Name    18300 non-null  object
dtypes: int64(1), object(3)
memory usage: 572.0+ KB


In [43]:
data_df.to_csv('clean_newsgroups.csv', index=False)

In [44]:
data_df = pd.read_csv('clean_newsgroups.csv')

In [45]:
from sklearn.model_selection import train_test_split

train_corpus, test_corpus, train_label_nums, test_label_nums, train_label_names, test_label_names =\
                                 train_test_split(np.array(data_df['Clean Article']), np.array(data_df['Target Label']),
                                                       np.array(data_df['Target Name']), test_size=0.33, random_state=42)

train_corpus.shape, test_corpus.shape

((12261,), (6039,))

In [46]:
from collections import Counter

trd = dict(Counter(train_label_names))
tsd = dict(Counter(test_label_names))

(pd.DataFrame([[key, trd[key], tsd[key]] for key in trd], 
             columns=['Target Label', 'Train Count', 'Test Count'])
.sort_values(by=['Train Count', 'Test Count'],
             ascending=False))

,Target Label,Train Count,Test Count
15,sci.crypt,667,295
0,soc.religion.christian,662,312
5,rec.motorcycles,660,309
10,comp.sys.ibm.pc.hardware,654,309
8,comp.windows.x,653,327
11,rec.sport.hockey,651,322
19,sci.space,649,304
7,sci.med,648,312
17,rec.sport.baseball,648,303
4,sci.electronics,647,309


In [47]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score

# build BOW features on train articles
cv = CountVectorizer(binary=False, min_df=0.0, max_df=1.0)
cv_train_features = cv.fit_transform(train_corpus)

In [48]:
# transform test articles into features
cv_test_features = cv.transform(test_corpus)

In [49]:
print('BOW model:> Train features shape:', cv_train_features.shape, ' Test features shape:', cv_test_features.shape)

BOW model:> Train features shape: (12261, 84122)  Test features shape: (6039, 84122)


In [50]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB(alpha=1)
mnb.fit(cv_train_features, train_label_names)
mnb_bow_cv_scores = cross_val_score(mnb, cv_train_features, train_label_names, cv=5)
mnb_bow_cv_mean_score = np.mean(mnb_bow_cv_scores)
print('CV Accuracy (5-fold):', mnb_bow_cv_scores)
print('Mean CV Accuracy:', mnb_bow_cv_mean_score)
mnb_bow_test_score = mnb.score(cv_test_features, test_label_names)
print('Test Accuracy:', mnb_bow_test_score)

CV Accuracy (5-fold): [0.67060742 0.66965742 0.67495922 0.67088091 0.65986949]
Mean CV Accuracy: 0.6691948933589327
Test Accuracy: 0.6797483026991223


In [51]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty='l2', max_iter=100, C=1, random_state=42)
lr.fit(cv_train_features, train_label_names)
lr_bow_cv_scores = cross_val_score(lr, cv_train_features, train_label_names, cv=5)
lr_bow_cv_mean_score = np.mean(lr_bow_cv_scores)
print('CV Accuracy (5-fold):', lr_bow_cv_scores)
print('Mean CV Accuracy:', lr_bow_cv_mean_score)
lr_bow_test_score = lr.score(cv_test_features, test_label_names)
print('Test Accuracy:', lr_bow_test_score)

CV Accuracy (5-fold): [0.67998369 0.67822186 0.68800979 0.68433931 0.67088091]
Mean CV Accuracy: 0.6802871138912367
Test Accuracy: 0.6870342771982116


In [52]:
from sklearn.svm import LinearSVC

svm = LinearSVC(penalty='l2', C=1, random_state=42)
svm.fit(cv_train_features, train_label_names)
svm_bow_cv_scores = cross_val_score(svm, cv_train_features, train_label_names, cv=5)
svm_bow_cv_mean_score = np.mean(svm_bow_cv_scores)
print('CV Accuracy (5-fold):', svm_bow_cv_scores)
print('Mean CV Accuracy:', svm_bow_cv_mean_score)
svm_bow_test_score = svm.score(cv_test_features, test_label_names)
print('Test Accuracy:', svm_bow_test_score)

CV Accuracy (5-fold): [0.62943335 0.6451876  0.6411093  0.6504894  0.64477977]
Mean CV Accuracy: 0.6421998830875267
Test Accuracy: 0.6540818016227852


In [53]:
from sklearn.linear_model import SGDClassifier

svm_sgd = SGDClassifier(loss='hinge', penalty='l2', max_iter=5, random_state=42)
svm_sgd.fit(cv_train_features, train_label_names)
svmsgd_bow_cv_scores = cross_val_score(svm_sgd, cv_train_features, train_label_names, cv=5)
svmsgd_bow_cv_mean_score = np.mean(svmsgd_bow_cv_scores)
print('CV Accuracy (5-fold):', svmsgd_bow_cv_scores)
print('Mean CV Accuracy:', svmsgd_bow_cv_mean_score)
svmsgd_bow_test_score = svm_sgd.score(cv_test_features, test_label_names)
print('Test Accuracy:', svmsgd_bow_test_score)

CV Accuracy (5-fold): [0.64777823 0.62805873 0.62438825 0.64559543 0.62846656]
Mean CV Accuracy: 0.6348574406010818
Test Accuracy: 0.6343765524093393


In [54]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=10, random_state=42)
rfc.fit(cv_train_features, train_label_names)
rfc_bow_cv_scores = cross_val_score(rfc, cv_train_features, train_label_names, cv=5)
rfc_bow_cv_mean_score = np.mean(rfc_bow_cv_scores)
print('CV Accuracy (5-fold):', rfc_bow_cv_scores)
print('Mean CV Accuracy:', rfc_bow_cv_mean_score)
rfc_bow_test_score = rfc.score(cv_test_features, test_label_names)
print('Test Accuracy:', rfc_bow_test_score)

CV Accuracy (5-fold): [0.53077864 0.51223491 0.54200653 0.50326264 0.50734095]
Mean CV Accuracy: 0.5191247325743554
Test Accuracy: 0.5272396092068223


In [55]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(n_estimators=10, random_state=42)
gbc.fit(cv_train_features, train_label_names)
gbc_bow_cv_scores = cross_val_score(gbc, cv_train_features, train_label_names, cv=5)
gbc_bow_cv_mean_score = np.mean(gbc_bow_cv_scores)
print('CV Accuracy (5-fold):', gbc_bow_cv_scores)
print('Mean CV Accuracy:', gbc_bow_cv_mean_score)
gbc_bow_test_score = gbc.score(cv_test_features, test_label_names)
print('Test Accuracy:', gbc_bow_test_score)

CV Accuracy (5-fold): [0.5527925  0.55587276 0.55954323 0.5497553  0.545677  ]
Mean CV Accuracy: 0.5527281572186802
Test Accuracy: 0.5505878456698129


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

# build BOW features on train articles
tv = TfidfVectorizer(use_idf=True, min_df=0.0, max_df=1.0)
tv_train_features = tv.fit_transform(train_corpus)

In [57]:
# transform test articles into features
tv_test_features = tv.transform(test_corpus)

In [58]:
print('TFIDF model:> Train features shape:', tv_train_features.shape, ' Test features shape:', tv_test_features.shape)

TFIDF model:> Train features shape: (12261, 84122)  Test features shape: (6039, 84122)


In [59]:
mnb = MultinomialNB(alpha=1)
mnb.fit(tv_train_features, train_label_names)
mnb_tfidf_cv_scores = cross_val_score(mnb, tv_train_features, train_label_names, cv=5)
mnb_tfidf_cv_mean_score = np.mean(mnb_tfidf_cv_scores)
print('CV Accuracy (5-fold):', mnb_tfidf_cv_scores)
print('Mean CV Accuracy:', mnb_tfidf_cv_mean_score)
mnb_tfidf_test_score = mnb.score(tv_test_features, test_label_names)
print('Test Accuracy:', mnb_tfidf_test_score)

CV Accuracy (5-fold): [0.70240522 0.70595432 0.71492659 0.69983687 0.71533442]
Mean CV Accuracy: 0.707691484076827
Test Accuracy: 0.7055803941049843


In [60]:
lr = LogisticRegression(penalty='l2', max_iter=100, C=1, random_state=42)
lr.fit(tv_train_features, train_label_names)
lr_tfidf_cv_scores = cross_val_score(lr, tv_train_features, train_label_names, cv=5)
lr_tfidf_cv_mean_score = np.mean(lr_tfidf_cv_scores)
print('CV Accuracy (5-fold):', lr_tfidf_cv_scores)
print('Mean CV Accuracy:', lr_tfidf_cv_mean_score)
lr_tfidf_test_score = lr.score(tv_test_features, test_label_names)
print('Test Accuracy:', lr_tfidf_test_score)

CV Accuracy (5-fold): [0.74072564 0.74143556 0.74836868 0.73694943 0.74388254]
Mean CV Accuracy: 0.7422723714810708
Test Accuracy: 0.7408511342937573


In [61]:
svm = LinearSVC(penalty='l2', C=1, random_state=42)
svm.fit(tv_train_features, train_label_names)
svm_tfidf_cv_scores = cross_val_score(svm, tv_train_features, train_label_names, cv=5)
svm_tfidf_cv_mean_score = np.mean(svm_tfidf_cv_scores)
print('CV Accuracy (5-fold):', svm_tfidf_cv_scores)
print('Mean CV Accuracy:', svm_tfidf_cv_mean_score)
svm_tfidf_test_score = svm.score(tv_test_features, test_label_names)
print('Test Accuracy:', svm_tfidf_test_score)

CV Accuracy (5-fold): [0.75173257 0.75734095 0.76468189 0.75652529 0.75244698]
Mean CV Accuracy: 0.7565455356792528
Test Accuracy: 0.7608875641662527


In [62]:
svm_sgd = SGDClassifier(loss='hinge', penalty='l2', max_iter=5, random_state=42)
svm_sgd.fit(tv_train_features, train_label_names)
svmsgd_tfidf_cv_scores = cross_val_score(svm_sgd, tv_train_features, train_label_names, cv=5)
svmsgd_tfidf_cv_mean_score = np.mean(svmsgd_tfidf_cv_scores)
print('CV Accuracy (5-fold):', svmsgd_tfidf_cv_scores)
print('Mean CV Accuracy:', svmsgd_tfidf_cv_mean_score)
svmsgd_tfidf_test_score = svm_sgd.score(tv_test_features, test_label_names)
print('Test Accuracy:', svmsgd_tfidf_test_score)

CV Accuracy (5-fold): [0.75377089 0.75163132 0.76223491 0.75897227 0.75897227]
Mean CV Accuracy: 0.757116331901078
Test Accuracy: 0.7608875641662527


In [63]:
rfc = RandomForestClassifier(n_estimators=10, random_state=42)
rfc.fit(tv_train_features, train_label_names)
rfc_tfidf_cv_scores = cross_val_score(rfc, tv_train_features, train_label_names, cv=5)
rfc_tfidf_cv_mean_score = np.mean(rfc_tfidf_cv_scores)
print('CV Accuracy (5-fold):', rfc_tfidf_cv_scores)
print('Mean CV Accuracy:', rfc_tfidf_cv_mean_score)
rfc_tfidf_test_score = rfc.score(tv_test_features, test_label_names)
print('Test Accuracy:', rfc_tfidf_test_score)

CV Accuracy (5-fold): [0.52425601 0.5322186  0.54812398 0.52569331 0.51712887]
Mean CV Accuracy: 0.5294841553007303
Test Accuracy: 0.515979466799139


In [64]:
gbc = GradientBoostingClassifier(n_estimators=10, random_state=42)
gbc.fit(tv_train_features, train_label_names)
gbc_tfidf_cv_scores = cross_val_score(gbc, tv_train_features, train_label_names, cv=5)
gbc_tfidf_cv_mean_score = np.mean(gbc_tfidf_cv_scores)
print('CV Accuracy (5-fold):', gbc_tfidf_cv_scores)
print('Mean CV Accuracy:', gbc_tfidf_cv_mean_score)
gbc_tfidf_test_score = gbc.score(tv_test_features, test_label_names)
print('Test Accuracy:', gbc_tfidf_test_score)

CV Accuracy (5-fold): [0.55646148 0.57096248 0.55791191 0.54486134 0.54730832]
Mean CV Accuracy: 0.5555011042841971
Test Accuracy: 0.5530717006126842


In [65]:
pd.DataFrame([['Naive Bayes', mnb_bow_cv_mean_score, mnb_bow_test_score, 
               mnb_tfidf_cv_mean_score, mnb_tfidf_test_score],
              ['Logistic Regression', lr_bow_cv_mean_score, lr_bow_test_score, 
               lr_tfidf_cv_mean_score, lr_tfidf_test_score],
              ['Linear SVM', svm_bow_cv_mean_score, svm_bow_test_score, 
               svm_tfidf_cv_mean_score, svm_tfidf_test_score],
              ['Linear SVM (SGD)', svmsgd_bow_cv_mean_score, svmsgd_bow_test_score, 
               svmsgd_tfidf_cv_mean_score, svmsgd_tfidf_test_score],
              ['Random Forest', rfc_bow_cv_mean_score, rfc_bow_test_score, 
               rfc_tfidf_cv_mean_score, rfc_tfidf_test_score],
              ['Gradient Boosted Machines', gbc_bow_cv_mean_score, gbc_bow_test_score, 
               gbc_tfidf_cv_mean_score, gbc_tfidf_test_score]],
             columns=['Model', 'CV Score (TF)', 'Test Score (TF)', 'CV Score (TF-IDF)', 'Test Score (TF-IDF)'],
             ).T

,0,1,2,3,4,5
Model,Naive Bayes,Logistic Regression,Linear SVM,Linear SVM (SGD),Random Forest,Gradient Boosted Machines
CV Score (TF),0.669195,0.680287,0.6422,0.634857,0.519125,0.552728
Test Score (TF),0.679748,0.687034,0.654082,0.634377,0.52724,0.550588
CV Score (TF-IDF),0.707691,0.742272,0.756546,0.757116,0.529484,0.555501
Test Score (TF-IDF),0.70558,0.740851,0.760888,0.760888,0.515979,0.553072


In [66]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

mnb_pipeline = Pipeline([('tfidf', TfidfVectorizer()),
                        ('mnb', MultinomialNB())
                       ])

param_grid = {'tfidf__ngram_range': [(1, 1), (1, 2)],
              'mnb__alpha': [1e-5, 1e-4, 1e-2, 1e-1, 1]
}

gs_mnb = GridSearchCV(mnb_pipeline, param_grid, cv=5, verbose=2)
gs_mnb = gs_mnb.fit(train_corpus, train_label_names)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 1); total time=   0.4s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 1); total time=   0.4s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 1); total time=   0.3s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 1); total time=   0.4s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 1); total time=   0.4s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 2); total time=   1.8s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 2); total time=   2.0s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 2); total time=   1.9s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 2); total time=   2.0s
[CV] END ........mnb__alpha=1e-05, tfidf__ngram_range=(1, 2); total time=   2.0s
[CV] END .......mnb__alpha=0.0001, tfidf__ngram_range=(1, 1); total time=   0.4s
[CV] END .......mnb__alpha=0.0001, tfidf__ngram_

In [67]:
gs_mnb.best_estimator_.get_params()

{'memory': None,
 'steps': [('tfidf', TfidfVectorizer()), ('mnb', MultinomialNB(alpha=0.01))],
 'transform_input': None,
 'verbose': False,
 'tfidf': TfidfVectorizer(),
 'mnb': MultinomialNB(alpha=0.01),
 'tfidf__analyzer': 'word',
 'tfidf__binary': False,
 'tfidf__decode_error': 'strict',
 'tfidf__dtype': numpy.float64,
 'tfidf__encoding': 'utf-8',
 'tfidf__input': 'content',
 'tfidf__lowercase': True,
 'tfidf__max_df': 1.0,
 'tfidf__max_features': None,
 'tfidf__min_df': 1,
 'tfidf__ngram_range': (1, 1),
 'tfidf__norm': 'l2',
 'tfidf__preprocessor': None,
 'tfidf__smooth_idf': True,
 'tfidf__stop_words': None,
 'tfidf__strip_accents': None,
 'tfidf__sublinear_tf': False,
 'tfidf__token_pattern': '(?u)\\b\\w\\w+\\b',
 'tfidf__tokenizer': None,
 'tfidf__use_idf': True,
 'tfidf__vocabulary': None,
 'mnb__alpha': 0.01,
 'mnb__class_prior': None,
 'mnb__fit_prior': True,
 'mnb__force_alpha': True}

In [68]:
cv_results = gs_mnb.cv_results_
results_df = pd.DataFrame({'rank': cv_results['rank_test_score'],
                           'params': cv_results['params'], 
                           'cv score (mean)': cv_results['mean_test_score'], 
                           'cv score (std)': cv_results['std_test_score']} 
              )
results_df = results_df.sort_values(by=['rank'], ascending=True)
pd.set_option('display.max_colwidth', 100)
results_df

,rank,params,cv score (mean),cv score (std)
4,1,"{'mnb__alpha': 0.01, 'tfidf__ngram_range': (1, 1)}",0.771634,0.008091
5,2,"{'mnb__alpha': 0.01, 'tfidf__ngram_range': (1, 2)}",0.770900,0.009750
6,3,"{'mnb__alpha': 0.1, 'tfidf__ngram_range': (1, 1)}",0.756791,0.007086
7,4,"{'mnb__alpha': 0.1, 'tfidf__ngram_range': (1, 2)}",0.752304,0.007796
3,5,"{'mnb__alpha': 0.0001, 'tfidf__ngram_range': (1, 2)}",0.751571,0.012029
2,6,"{'mnb__alpha': 0.0001, 'tfidf__ngram_range': (1, 1)}",0.744231,0.010717
1,7,"{'mnb__alpha': 1e-05, 'tfidf__ngram_range': (1, 2)}",0.742191,0.012109
0,8,"{'mnb__alpha': 1e-05, 'tfidf__ngram_range': (1, 1)}",0.729795,0.008310
8,9,"{'mnb__alpha': 1, 'tfidf__ngram_range': (1, 1)}",0.708752,0.006318
9,10,"{'mnb__alpha': 1, 'tfidf__ngram_range': (1, 2)}",0.701411,0.004842


In [69]:
best_mnb_test_score = gs_mnb.score(test_corpus, test_label_names)
print('Test Accuracy :', best_mnb_test_score)

Test Accuracy : 0.7752939228349064


In [70]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

In [71]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

lr_pipeline = Pipeline([('tfidf', TfidfVectorizer()),
                        ('lr', LogisticRegression(penalty='l2', max_iter=100, random_state=42))
                       ])

param_grid = {'tfidf__ngram_range': [(1, 1), (1, 2)],
              'lr__C': [1, 5, 10]
}

gs_lr = GridSearchCV(lr_pipeline, param_grid, cv=5, verbose=2)
gs_lr = gs_lr.fit(train_corpus, train_label_names)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 1); total time=   6.4s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 1); total time=   6.3s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 1); total time=   5.5s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 1); total time=   5.3s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 1); total time=   6.9s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 2); total time=  40.3s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 2); total time=  48.9s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 2); total time= 2.1min
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 2); total time=  34.3s
[CV] END .................lr__C=1, tfidf__ngram_range=(1, 2); total time=  37.7s
[CV] END .................lr__C=5, tfidf__ngram_range=(1, 1); total time=  25.5s
[CV] END .................lr__C=5, tfidf__ngram_r

In [72]:
gs_lr.best_estimator_

,steps,"[('tfidf', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [73]:
best_lr_test_score = gs_lr.score(test_corpus, test_label_names)
print('Test Accuracy :', best_lr_test_score)

Test Accuracy : 0.7605563835072032


In [74]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

svm_pipeline = Pipeline([('tfidf', TfidfVectorizer()),
                        ('svm', LinearSVC(random_state=42))
                       ])

param_grid = {'tfidf__ngram_range': [(1, 1), (1, 2)],
              'svm__C': [0.01, 0.1, 1, 5]
}

gs_svm = GridSearchCV(svm_pipeline, param_grid, cv=5, verbose=2)
gs_svm = gs_svm.fit(train_corpus, train_label_names)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 2); total time=   2.1s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 2); total time=   2.3s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 2); total time=   2.2s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 2); total time=   2.3s
[CV] END .............svm__C=0.01, tfidf__ngram_range=(1, 2); total time=   2.2s
[CV] END ..............svm__C=0.1, tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END ..............svm__C=0.1, tfidf__ngram_r

In [75]:
gs_svm.best_estimator_.get_params()

{'memory': None,
 'steps': [('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
  ('svm', LinearSVC(C=5, random_state=42))],
 'transform_input': None,
 'verbose': False,
 'tfidf': TfidfVectorizer(ngram_range=(1, 2)),
 'svm': LinearSVC(C=5, random_state=42),
 'tfidf__analyzer': 'word',
 'tfidf__binary': False,
 'tfidf__decode_error': 'strict',
 'tfidf__dtype': numpy.float64,
 'tfidf__encoding': 'utf-8',
 'tfidf__input': 'content',
 'tfidf__lowercase': True,
 'tfidf__max_df': 1.0,
 'tfidf__max_features': None,
 'tfidf__min_df': 1,
 'tfidf__ngram_range': (1, 2),
 'tfidf__norm': 'l2',
 'tfidf__preprocessor': None,
 'tfidf__smooth_idf': True,
 'tfidf__stop_words': None,
 'tfidf__strip_accents': None,
 'tfidf__sublinear_tf': False,
 'tfidf__token_pattern': '(?u)\\b\\w\\w+\\b',
 'tfidf__tokenizer': None,
 'tfidf__use_idf': True,
 'tfidf__vocabulary': None,
 'svm__C': 5,
 'svm__class_weight': None,
 'svm__dual': 'auto',
 'svm__fit_intercept': True,
 'svm__intercept_scaling': 1,
 'svm__loss': 'squa

In [76]:
best_svm_test_score = gs_svm.score(test_corpus, test_label_names)
print('Test Accuracy :', best_svm_test_score)

Test Accuracy : 0.7738036098691836


In [77]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

sgd_pipeline = Pipeline([('tfidf', TfidfVectorizer()),
                        ('sgd', SGDClassifier(random_state=42))
                       ])

param_grid = {'tfidf__ngram_range': [(1, 1), (1, 2)],
              'sgd__alpha': [1e-7, 1e-6, 1e-5, 1e-4]
}

gs_sgd = GridSearchCV(sgd_pipeline, param_grid, cv=5, verbose=2)
gs_sgd = gs_sgd.fit(train_corpus, train_label_names)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 2); total time=   2.6s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 2); total time=   2.6s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 2); total time=   2.7s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 2); total time=   2.8s
[CV] END ........sgd__alpha=1e-07, tfidf__ngram_range=(1, 2); total time=   2.6s
[CV] END ........sgd__alpha=1e-06, tfidf__ngram_range=(1, 1); total time=   0.6s
[CV] END ........sgd__alpha=1e-06, tfidf__ngram_r

In [78]:
gs_sgd.best_estimator_.get_params()

{'memory': None,
 'steps': [('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
  ('sgd', SGDClassifier(random_state=42))],
 'transform_input': None,
 'verbose': False,
 'tfidf': TfidfVectorizer(ngram_range=(1, 2)),
 'sgd': SGDClassifier(random_state=42),
 'tfidf__analyzer': 'word',
 'tfidf__binary': False,
 'tfidf__decode_error': 'strict',
 'tfidf__dtype': numpy.float64,
 'tfidf__encoding': 'utf-8',
 'tfidf__input': 'content',
 'tfidf__lowercase': True,
 'tfidf__max_df': 1.0,
 'tfidf__max_features': None,
 'tfidf__min_df': 1,
 'tfidf__ngram_range': (1, 2),
 'tfidf__norm': 'l2',
 'tfidf__preprocessor': None,
 'tfidf__smooth_idf': True,
 'tfidf__stop_words': None,
 'tfidf__strip_accents': None,
 'tfidf__sublinear_tf': False,
 'tfidf__token_pattern': '(?u)\\b\\w\\w+\\b',
 'tfidf__tokenizer': None,
 'tfidf__use_idf': True,
 'tfidf__vocabulary': None,
 'sgd__alpha': 0.0001,
 'sgd__average': False,
 'sgd__class_weight': None,
 'sgd__early_stopping': False,
 'sgd__epsilon': 0.1,
 'sgd__eta0': 0.

In [79]:
best_sgd_test_score = gs_sgd.score(test_corpus, test_label_names)
print('Test Accuracy :', best_sgd_test_score)

Test Accuracy : 0.7724788872329856


In [80]:
import model_evaluation_utils as meu

In [81]:
mnb_predictions = gs_mnb.predict(test_corpus)
unique_classes = list(set(test_label_names))
meu.get_metrics(true_labels=test_label_names, predicted_labels=mnb_predictions)

Accuracy: 0.7753
Precision: 0.7808
Recall: 0.7753
F1 Score: 0.7726


In [82]:
meu.display_classification_report(true_labels=test_label_names, 
                                  predicted_labels=mnb_predictions, classes=unique_classes)

                          precision    recall  f1-score   support

 comp.os.ms-windows.misc       0.74      0.69      0.71       304
          comp.windows.x       0.87      0.81      0.84       327
            misc.forsale       0.80      0.68      0.73       319
   comp.sys.mac.hardware       0.81      0.76      0.79       315
      talk.politics.misc       0.65      0.67      0.66       256
  soc.religion.christian       0.69      0.88      0.77       312
      talk.politics.guns       0.72      0.82      0.76       314
           comp.graphics       0.67      0.74      0.70       307
         rec.motorcycles       0.78      0.78      0.78       309
comp.sys.ibm.pc.hardware       0.63      0.76      0.69       309
        rec.sport.hockey       0.94      0.92      0.93       322
         sci.electronics       0.70      0.72      0.71       309
               rec.autos       0.85      0.77      0.81       343
               sci.crypt       0.76      0.85      0.80       295
      rec

In [83]:
label_data_map = {v:k for k, v in data_labels_map.items()}
label_map_df = pd.DataFrame(list(label_data_map.items()), columns=['Label Name', 'Label Number'])
label_map_df

,Label Name,Label Number
0,alt.atheism,0
1,comp.graphics,1
2,comp.os.ms-windows.misc,2
3,comp.sys.ibm.pc.hardware,3
4,comp.sys.mac.hardware,4
5,comp.windows.x,5
6,misc.forsale,6
7,rec.autos,7
8,rec.motorcycles,8
9,rec.sport.baseball,9


In [84]:
unique_class_nums = label_map_df['Label Number'].values
mnb_prediction_class_nums = [label_data_map[item] for item in mnb_predictions]
meu.display_confusion_matrix_pretty(true_labels=test_label_nums, 
                                   predicted_labels=mnb_prediction_class_nums, classes=unique_class_nums)

Predicted:                                                         \
                   0    1    2    3    4    5    6    7    8    9    10   11   
Actual: 0         173    2    0    2    0    1    0    2    5    0    3    4   
        1           2  226   10   13    8   15    4    0    1    2    1    8   
        2           0   14  209   42   10   12    5    0    0    0    0    3   
        3           0   11   26  235   10    2    7    0    0    0    0    2   
        4           0   10    5   25  240    2   10    1    0    0    0    7   
        5           0   30   12    4    4  266    1    0    2    0    0    2   
        6           0    4    7   27   12    1  216   13    3    2    0    7   
        7           0    0    2    3    4    1    7  264   26    0    2    2   
        8           1    1    0    1    1    1    4   17  240    2    3    3   
        9           2    2    1    1    0    1    2    0    3  274    8    1   
        10          2    1    1    1    0    1    0    0    1   10  296    0   
        11          2    4    1    0    0    1    1    0    3    0    0  251   
        12          1   13    3   17    6    0   10    5    3    2    0   11   
        13          2    4    1    0    0    0    1    1    4    1    0    2   
        14          5   11    2    0    0    1    2    2    2    0    1    3   
        15         14    1    1    1    1    0    0    0    0    0    0    2   
        16          2    0    2    0    0    0    1    2    1    1    1    8   
        17          6    0    0    1    0    0    0    2    4    0    1    6   
        18          6    2    0    0    0    0    0    2    5    1    0    9   
        19         33    1    0    0    0    1    0    1    4    2    0    1   

                                                   
             12   13   14   15   16   17   18  19  
Actual: 0     1    1    2   32    6   12   13   8  
        1     6    3    5    0    2    0    1   0  
        2     4    0    3    0    0    0    2   0  
        3    16    0    0    0    0    0    0   0  
        4    13    1    1    0    0    0    0   0  
        5     3    0    1    0    1    1    0   0  
        6    17    1    5    2    2    0    0   0  
        7    11    2    3    3    5    1    7   0  
        8     4    3    3    5   10    2    8   0  
        9     1    0    0    2    0    2    3   0  
        10    0    2    1    1    0    1    4   0  
        11    6    0    3    2   11    3    6   1  
        12  224    4    7    0    3    0    0   0  
        13    3  276    8    4    1    1    3   0  
        14    5    3  258    1    3    2    3   0  
        15    2    3    1  274    5    2    3   2  
        16    1    1    4    5  256    5   21   3  
        17    0    0    2    4    5  269   11   0  
        18    1    9    7    2   31    9  171   1  
        19    0    5    3   61   15    4    6  64

In [85]:
unique_classes = label_map_df['Label Name'].values
meu.display_confusion_matrix_pretty(true_labels=test_label_names, 
                                    predicted_labels=mnb_predictions, classes=unique_classes)

Predicted:                \
                                 alt.atheism comp.graphics   
Actual: alt.atheism                      173             2   
        comp.graphics                      2           226   
        comp.os.ms-windows.misc            0            14   
        comp.sys.ibm.pc.hardware           0            11   
        comp.sys.mac.hardware              0            10   
        comp.windows.x                     0            30   
        misc.forsale                       0             4   
        rec.autos                          0             0   
        rec.motorcycles                    1             1   
        rec.sport.baseball                 2             2   
        rec.sport.hockey                   2             1   
        sci.crypt                          2             4   
        sci.electronics                    1            13   
        sci.med                            2             4   
        sci.space                          5            11   
        soc.religion.christian            14             1   
        talk.politics.guns                 2             0   
        talk.politics.mideast              6             0   
        talk.politics.misc                 6             2   
        talk.religion.misc                33             1   

                                                          \
                                 comp.os.ms-windows.misc   
Actual: alt.atheism                                    0   
        comp.graphics                                 10   
        comp.os.ms-windows.misc                      209   
        comp.sys.ibm.pc.hardware                      26   
        comp.sys.mac.hardware                          5   
        comp.windows.x                                12   
        misc.forsale                                   7   
        rec.autos                                      2   
        rec.motorcycles                                0   
        rec.sport.baseball                             1   
        rec.sport.hockey                               1   
        sci.crypt                                      1   
        sci.electronics                                3   
        sci.med                                        1   
        sci.space                                      2   
        soc.religion.christian                         1   
        talk.politics.guns                             2   
        talk.politics.mideast                          0   
        talk.politics.misc                             0   
        talk.religion.misc                             0   

                                                           \
                                 comp.sys.ibm.pc.hardware   
Actual: alt.atheism                                     2   
        comp.graphics                                  13   
        comp.os.ms-windows.misc                        42   
        comp.sys.ibm.pc.hardware                      235   
        comp.sys.mac.hardware                          25   
        comp.windows.x                                  4   
        misc.forsale                                   27   
        rec.autos                                       3   
        rec.motorcycles                                 1   
        rec.sport.baseball                              1   
        rec.sport.hockey                                1   
        sci.crypt                                       0   
        sci.electronics                                17   
        sci.med                                         0   
        sci.space                                       0   
        soc.religion.christian                          1   
        talk.politics.guns                              0   
        talk.politics.mideast                           1   
        talk.politics.misc                              0   
        talk.religion.misc                              0   

     

In [86]:
label_map_df[label_map_df['Label Number'].isin([0, 15, 19])]

,Label Name,Label Number
0,alt.atheism,0
15,soc.religion.christian,15
19,talk.religion.misc,19


In [87]:
train_idx, test_idx = train_test_split(np.array(range(len(data_df['Article']))), test_size=0.33, random_state=42)
test_idx

array([ 4097,  8528,  7621, ..., 14979,  4772,  7800])

In [88]:
predict_probas = gs_mnb.predict_proba(test_corpus).max(axis=1)
test_df = data_df.iloc[test_idx]
test_df['Predicted Name'] = mnb_predictions
test_df['Predicted Confidence'] = predict_probas
test_df.head()

,Article,Clean Article,Target Label,Target Name,Predicted Name,Predicted Confidence
4097,\nDid you watch the games????\n\n,watch game,10,rec.sport.hockey,rec.sport.hockey,0.531198
8528,I too have been watching the IIsi speedup reports and plan to upgrade in\nthe next few weeks. T...,watch iisi speedup report plan upgrade next week plan build small board different crystal able s...,4,comp.sys.mac.hardware,comp.sys.mac.hardware,0.556093
7621,"\nI think one (not ideal) solution is to use the\ntracing utility (can't remember the name, sorr...",think one not ideal solution use tracing utility not remember name sorry include corel draw w pa...,1,comp.graphics,comp.graphics,0.986678
4754,\n I am curious about knowing which commericial cars today\n have v engines.\n\n V4 - I ...,curious know commericial car today v engine v not know v legend mr mr vw golf passat l vr inline...,7,rec.autos,rec.autos,0.999901
15903,"DH>>Does anyone out their have a mountain tape backup that I could compare\nDH>>notes with, (jum...",dhdoe anyone mountain tape backup could compare dhnote jumper setting software ect dhor anyone k...,3,comp.sys.ibm.pc.hardware,comp.sys.ibm.pc.hardware,0.362778


In [89]:
pd.set_option('display.max_colwidth', 200)
res_df = (test_df[(test_df['Target Name'] == 'talk.religion.misc') & (test_df['Predicted Name'] == 'soc.religion.christian')]
       .sort_values(by=['Predicted Confidence'], ascending=False).head(5))
res_df

,Article,Clean Article,Target Label,Target Name,Predicted Name,Predicted Confidence
4304,"\nOK, here's at least one Christian's answer:\n\nJesus was a JEW, not a Christian. In this context Matthew 5:14-19 makes\nsense. Matt 5:17 ""Do not think that I [Jesus] came to abolish the Law or...",ok least one christians answer jesus jew not christian context matthew make sense matt not think jesus come abolish law prophets not come abolish fulfill jesus live jewish law however culmination ...,19,talk.religion.misc,soc.religion.christian,0.993241
4237,"The Nicene Creed\n\nWE BELIEVE in one God the Father Almighty, Maker of heaven and earth, and of all things visible and invisible.\nAnd in one Lord Jesus Christ, the only-begotten Son of God, bego...",nicene creed believe one god father almighty maker heaven earth thing visible invisible one lord jesus christ beget son god beget father world god god light light god god beget not make one substa...,19,talk.religion.misc,soc.religion.christian,0.991746
14513,"iank@microsoft.com (Ian Kennedy) writes...\n\n\nMore along the lines of Hebrews 12:25-29, I reckon...\n\n\tSee that you refuse not him that speaks. For if they\n\tescaped not who refused him that ...",iankmicrosoft com ian kennedy write along line hebrews reckon see refuse not speak escape not refuse spake earth much shall not escape turn away speak heaven whose voice shake earth promise say ye...,19,talk.religion.misc,soc.religion.christian,0.990353
16678,"\nJesus did not say that he was the fulfillment of the Law, and, unless\nI'm mistaken, heaven and earth have not yet passed away. Am I mistaken?\nAnd, even assuming that one can just gloss over th...",jesus not say fulfillment law unless mistaken heaven earth not yet pass away mistaken even assume one gloss portion word jesus really think accomplish not jesus say jew annul v say jesus record wo...,19,talk.religion.misc,soc.religion.christian,0.987394
13764,": \n: I am a Mormon. I believe in Christ, that he is alive. He raised himself\n: [Text deleted]\n:\n: I learned that the concept of the Holy Trinity was never taught by Jesus\n: Christ, that it ...",mormon believe christ alive raise text delete learn concept holy trinity never teach jesus christ agree council clergyman long christ ascend man no authority speak jesus never teach concept trinit...,19,talk.religion.misc,soc.religion.christian,0.977881


In [90]:
pd.set_option('display.max_colwidth', 200)
res_df = (test_df[(test_df['Target Name'] == 'talk.religion.misc') & (test_df['Predicted Name'] == 'alt.atheism')]
       .sort_values(by=['Predicted Confidence'], ascending=False).head(5))
res_df

,Article,Clean Article,Target Label,Target Name,Predicted Name,Predicted Confidence
4706,"This discussion on ""objective"" seems to be falling into solipsism (Eg: the\nrecent challenge from Frank Dwyer, for someone to prove that he can actually\nobserve phenomena). Someones even made th...",discussion objective seem fall solipsism eg recent challenge frank dwyer someone prove actually observe phenomena someones even make statement science subjective even atom subjective get bit silly...,19,talk.religion.misc,alt.atheism,0.973821
914,"\n\nAtoms are not objective. They aren't even real. What scientists call\nan atom is nothing more than a mathematical model that describes \ncertain physical, observable properties of our surrou...",atom not objective not even real scientist call atom nothing mathematical model describe certain physical observable property surrounding subjective objective though approach scientist take discus...,19,talk.religion.misc,alt.atheism,0.932306
11820,\nI think that if a theist were truly objective and throws out the notion that\nGod definitely exists and starts from scratch to prove to themselves that\nthe scriptures are the whole truth then t...,think theist truly objective throw notion god definitely exist start scratch prove scripture whole truth person would no long theist miss something people convert non theism theism bring non theis...,19,talk.religion.misc,alt.atheism,0.827296
6020,"\n\n[""it"" is Big Bang]\n\nSince you asked... from the Big Bang to the formation of atoms is about\n10E11 seconds. As for the ""color"": bright. Very very bright. \n\n\nI don't. I believe the curren...",big bang since ask big bang formation atom e second color bright bright not believe current theory cosmology fairly well support observational evidence not well support say evolution relativity an...,19,talk.religion.misc,alt.atheism,0.790560
2334,In <1ren9a$94q@morrow.stanford.edu> salem@pangea.Stanford.EDU (Bruce Salem) \n\n\n\nThis brings up another something I have never understood. I asked this once\nbefore and got a few interesting r...,renaqmorrow stanford edu salempangea stanford edu bruce salem bring another something never understand ask get interesting response somehow not seem satisfied would nt not consider good source mig...,19,talk.religion.misc,alt.atheism,0.758135
